# Frequency Band Ablation Study — 4 DL Models
TCN, 1D-CNN, EEGNet, EEG-Conformer

依次剔除各频段，观察 AUC 变化（ΔAUC）。不重新训练，只修改测试集输入。

In [ ]:
import os, sys, json
import numpy as np
import torch
import torch.nn as nn
from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import roc_auc_score

sys.path.insert(0, r'D:\Code')
from data_utils import get_dataloaders_for_seed, DATA_DIR

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FS         = 256
N_CHANNELS = 18
WIN        = 20 * FS
SEED       = 42

WEIGHTS = {
    'TCN':           r'D:\seizure_results\tcn\seed42_best.pt',
    '1D-CNN':        r'D:\seizure_results\1dcnn\seed42_best.pt',
    'EEGNet':        r'D:\seizure_results\eegnet\seed42_best.pt',
    'EEG-Conformer': r'D:\seizure_results\eeg_conformer\seed42_best.pt',
}
OUT_DIR = r'D:\seizure_results\ablation'

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')

Device: cuda


In [ ]:
# Model definitions 

# TCN
class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=padding))
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=padding))
        nn.init.normal_(self.conv1.weight_v, 0, 0.01); nn.init.ones_(self.conv1.weight_g)
        nn.init.normal_(self.conv2.weight_v, 0, 0.01); nn.init.ones_(self.conv2.weight_g)
        self.net = nn.Sequential(self.conv1, nn.ReLU(), nn.Dropout(dropout),
                                 self.conv2, nn.ReLU(), nn.Dropout(dropout))
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        return self.relu(out[:, :, :x.size(2)] + self.residual(x))

class TCN(nn.Module):
    def __init__(self, n_channels=N_CHANNELS, n_filters=64, kernel_size=8,
                 dilations=(1,2,4,8), dropout=0.2, n_classes=2):
        super().__init__()
        layers, in_ch = [], n_channels
        for d in dilations:
            layers.append(TemporalBlock(in_ch, n_filters, kernel_size, d, dropout))
            in_ch = n_filters
        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(n_filters, 64),
                                        nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        return self.classifier(self.pool(self.tcn(x)))


# 1D-CNN 
class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernels=(3,5,7)):
        super().__init__()
        branch_ch = out_ch // len(kernels)
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv1d(in_ch, branch_ch, k, padding=k//2, bias=False),
                          nn.BatchNorm1d(branch_ch), nn.GELU()) for k in kernels])
    def forward(self, x):
        return torch.cat([b(x) for b in self.branches], dim=1)

class CNN1D(nn.Module):
    def __init__(self, n_channels=N_CHANNELS, dropout=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            MultiScaleBlock(n_channels, 96), nn.MaxPool1d(4), nn.Dropout(dropout*0.5),
            MultiScaleBlock(96, 192),        nn.MaxPool1d(4), nn.Dropout(dropout*0.5),
            MultiScaleBlock(192, 384),       nn.AdaptiveAvgPool1d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(384, 128),
                                        nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 2))
    def forward(self, x):
        return self.classifier(self.encoder(x))


# EEGNet
class EEGNet(nn.Module):
    def __init__(self, n_channels=N_CHANNELS, n_times=WIN, F1=8, D=2,
                 kern_length=127, dropout=0.5, n_classes=2):
        super().__init__()
        F2 = F1 * D
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kern_length), padding=(0, kern_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1*D, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D), nn.ELU(), nn.AvgPool2d(kernel_size=(1,4)), nn.Dropout(dropout))
        self.block2 = nn.Sequential(
            nn.Conv2d(F2, F2, kernel_size=(1,15), padding=(0,7), groups=F2, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1,1), bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d(kernel_size=(1,8)), nn.Dropout(dropout))
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_times)
            flat  = self.block2(self.block1(dummy)).view(1,-1).shape[1]
        self.classifier = nn.Linear(flat, n_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.block2(self.block1(x))
        return self.classifier(x.view(x.size(0), -1))


# EEG-Conformer
class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=N_CHANNELS, emb_size=40, kern_t=15,
                 pool_size=75, pool_stride=75, dropout=0.5):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, emb_size, kernel_size=(1, kern_t), padding=(0, kern_t//2), bias=False),
            nn.BatchNorm2d(emb_size))
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(emb_size, emb_size, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(emb_size), nn.ELU(), nn.Dropout(dropout))
        self.pool = nn.AvgPool2d(kernel_size=(1, pool_size), stride=(1, pool_stride))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x).squeeze(2)
        return x.permute(0, 2, 1)

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout=0.5):
        super().__init__()
        self.attn    = nn.MultiheadAttention(emb_size, num_heads, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.attn(x, x, x)
        return self.dropout(out)

class FeedForward(nn.Module):
    def __init__(self, emb_size, expansion=4, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_size, emb_size*expansion), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(emb_size*expansion, emb_size), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, emb_size, num_heads, dropout=0.5):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_size); self.attn = MultiHeadAttention(emb_size, num_heads, dropout)
        self.norm2 = nn.LayerNorm(emb_size); self.ff   = FeedForward(emb_size, dropout=dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ff(self.norm2(x))

class TransformerEncoder(nn.Module):
    def __init__(self, depth, emb_size, num_heads, dropout=0.5):
        super().__init__()
        self.layers = nn.ModuleList([TransformerEncoderBlock(emb_size, num_heads, dropout) for _ in range(depth)])
    def forward(self, x):
        for layer in self.layers: x = layer(x)
        return x

class ClassificationHead(nn.Module):
    def __init__(self, emb_size, n_tokens, n_classes=2, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(emb_size*n_tokens, 256), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(256, 32), nn.ELU(), nn.Dropout(dropout), nn.Linear(32, n_classes))
    def forward(self, x): return self.net(x)

class EEGConformer(nn.Module):
    def __init__(self, n_channels=N_CHANNELS, n_times=WIN, emb_size=40, depth=6,
                 num_heads=10, kern_t=15, pool_size=75, pool_stride=75,
                 dropout=0.5, n_classes=2):
        super().__init__()
        self.patch_embed = PatchEmbedding(n_channels, emb_size, kern_t, pool_size, pool_stride, dropout)
        with torch.no_grad():
            n_tokens = self.patch_embed(torch.zeros(1, n_channels, n_times)).shape[1]
        self.transformer = TransformerEncoder(depth, emb_size, num_heads, dropout)
        self.classifier  = ClassificationHead(emb_size, n_tokens, n_classes, dropout)
    def forward(self, x):
        return self.classifier(self.transformer(self.patch_embed(x)))


print('All model classes defined.')

All model classes defined.


In [ ]:
# Load test data 
print(f'Loading test data for seed {SEED}...')
_, _, test_loader, _, _, _, _, _ = get_dataloaders_for_seed(SEED, DATA_DIR, batch_size=256)

X_list, y_list = [], []
for x, y in test_loader:
    X_list.append(x.numpy())
    y_list.append(y.numpy())

X_test = np.concatenate(X_list, axis=0)  # (N, 18, 5120)
y_test = np.concatenate(y_list, axis=0)  # (N,)
print(f'Test set: {X_test.shape}  pre={(y_test==1).sum()}  inter={(y_test==0).sum()}')

Loading test data for seed 42...

[seed=42]  train=15pts  val=4pts  test=4pts
  train  total=11726  pre=5074  inter=6652  ratio=1:1
  val    total=3521  pre=2084  inter=1437
  test   total=2177  pre=1334  inter=843
Test set: (2177, 18, 5120)  pre=1334  inter=843


In [ ]:
# Band-stop filter
{
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 40.0),
}

def bandstop_filter(X, low, high, fs=FS, order=4):
    nyq = fs / 2.0
    lo  = max(low, 0.1) / nyq
    hi  = min(high, nyq * 0.99) / nyq
    sos = butter(order, [lo, hi], btype='bandstop', output='sos')
    X_filt = X.copy()
    for i in range(X_filt.shape[0]):
        X_filt[i] = sosfiltfilt(sos, X_filt[i], axis=-1).astype(np.float32)
    return X_filt

print('Filter helper defined.')

Filter helper defined.


In [ ]:
# Inference helper
def get_probs(model, X_np, batch_size=256):
    model.eval()
    probs = []
    for i in range(0, len(X_np), batch_size):
        batch = torch.from_numpy(X_np[i:i+batch_size]).float().to(DEVICE)
        p = torch.softmax(model(batch), dim=1)[:, 1].cpu().numpy()
        probs.append(p)
    return np.concatenate(probs)

print('Inference helper defined.')

Inference helper defined.


In [ ]:
# Load all models 
MODEL_CLASSES = {
    'TCN':           TCN,
    '1D-CNN':        CNN1D,
    'EEGNet':        EEGNet,
    'EEG-Conformer': EEGConformer,
}

models = {}
for name, cls in MODEL_CLASSES.items():
    m = cls().to(DEVICE)
    m.load_state_dict(torch.load(WEIGHTS[name], map_location=DEVICE))
    m.eval()
    models[name] = m
    print(f'  Loaded {name}')

c:\Users\11217\anaconda3\envs\seizure_prediction\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Loaded TCN
  Loaded 1D-CNN
  Loaded EEGNet
  Loaded EEG-Conformer


In [ ]:
# Run ablation
results = {}

for model_name, model in models.items():
    print(f'\n=== {model_name} ===')

    base_probs = get_probs(model, X_test)
    base_auc   = roc_auc_score(y_test, base_probs)
    print(f'  Baseline AUC: {base_auc:.4f}')

    model_results = {'baseline': base_auc}
    for band_name, (low, high) in BANDS.items():
        print(f'  Ablating {band_name} ({low}-{high} Hz)...', end=' ')
        X_abl     = bandstop_filter(X_test, low, high)
        abl_auc   = roc_auc_score(y_test, get_probs(model, X_abl))
        delta     = base_auc - abl_auc
        print(f'AUC={abl_auc:.4f}  ΔAUC={delta:+.4f}')
        model_results[band_name] = {'auc': abl_auc, 'delta': delta}

    results[model_name] = model_results

out_path = os.path.join(OUT_DIR, 'ablation_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to {out_path}')


=== TCN ===
  Baseline AUC: 0.5974
  Ablating delta (0.5-4.0 Hz)... AUC=0.5871  ΔAUC=+0.0104
  Ablating theta (4.0-8.0 Hz)... AUC=0.6053  ΔAUC=-0.0079
  Ablating alpha (8.0-13.0 Hz)... AUC=0.6091  ΔAUC=-0.0117
  Ablating beta (13.0-30.0 Hz)... AUC=0.5704  ΔAUC=+0.0271
  Ablating gamma (30.0-40.0 Hz)... AUC=0.5959  ΔAUC=+0.0016

=== 1D-CNN ===
  Baseline AUC: 0.5959
  Ablating delta (0.5-4.0 Hz)... AUC=0.5997  ΔAUC=-0.0038
  Ablating theta (4.0-8.0 Hz)... AUC=0.6054  ΔAUC=-0.0095
  Ablating alpha (8.0-13.0 Hz)... AUC=0.6006  ΔAUC=-0.0047
  Ablating beta (13.0-30.0 Hz)... AUC=0.5782  ΔAUC=+0.0178
  Ablating gamma (30.0-40.0 Hz)... AUC=0.6146  ΔAUC=-0.0186

=== EEGNet ===
  Baseline AUC: 0.5641
  Ablating delta (0.5-4.0 Hz)... AUC=0.5614  ΔAUC=+0.0027
  Ablating theta (4.0-8.0 Hz)... AUC=0.5997  ΔAUC=-0.0357
  Ablating alpha (8.0-13.0 Hz)... AUC=0.5665  ΔAUC=-0.0024
  Ablating beta (13.0-30.0 Hz)... AUC=0.5115  ΔAUC=+0.0526
  Ablating gamma (30.0-40.0 Hz)... AUC=0.5657  ΔAUC=-0.0017

===

In [ ]:
# Summary table
model_names = list(models.keys())

header = f'{"Band":<10}' + ''.join(f'{n:>16}' for n in model_names)
print('\nFrequency Band Ablation — ΔAUC (positive = band is important)')
print(header)
print('-' * (10 + 16 * len(model_names)))

for band in BANDS:
    row = f'{band:<10}'
    for n in model_names:
        d = results[n][band]['delta']
        row += f'{d:>+16.4f}'
    print(row)

print('-' * (10 + 16 * len(model_names)))
base_row = f'{"Baseline":<10}'
for n in model_names:
    base_row += f'{results[n]["baseline"]:>16.4f}'
print(base_row)


Frequency Band Ablation — ΔAUC (positive = band is important)
Band                   TCN          1D-CNN          EEGNet   EEG-Conformer
--------------------------------------------------------------------------
delta              +0.0104         -0.0038         +0.0027         +0.0122
theta              -0.0079         -0.0095         -0.0357         -0.0126
alpha              -0.0117         -0.0047         -0.0024         -0.0115
beta               +0.0271         +0.0178         +0.0526         -0.0201
gamma              +0.0016         -0.0186         -0.0017         -0.0035
--------------------------------------------------------------------------
Baseline            0.5974          0.5959          0.5641          0.4907
